# SCFF on LeNet-5 / MNIST: results from the stored weights

This notebook scores the stored weight files on the noisy MNIST test set and writes the results to an Excel workbook.
It trains nothing and reads no other notebook, script or results file: everything it needs is in the cells below.

| Row | Weight file, for input-noise variance *v* |
|---|---|
| Init (no retrain) | `weights_baselines/full_retrain_var_0.0.pth`, the clean model every run starts from |
| SCFF | `weights_scff_lenet5_mnist/scff_var_<v>.pth`, an SCFF network state dict (the LeNet-5 conv body plus SCFF's linear readout), scored with that network |

- **Test set:** the official MNIST test set, 10,000 images, no re-split. Gaussian noise of variance *v* is added after
  normalisation, drawn on the CPU with seed 42 at every level.
- **Metrics:** accuracy, macro precision, recall and F1, and mAP (the mean one-vs-rest average precision on the softmax
  scores), rounded to 6 decimals. Accuracies can differ by a few test images between machines (GPU and CPU arithmetic).
- **Output:** `lenet5_mnist/results_scff_lenet5_mnist/results_scff_lenet5_mnist_reproduced.xlsx`, one sheet `long`
  with a row per noise level and method; a re-run overwrites it. The name carries no `YYYYmmdd_HHMMSS` stamp, so the
  plotting notebooks, which read the workbook with the newest stamp in each results folder, keep using the reported one.

**How to run:** run all cells. The notebook looks for the project folder (the one holding `lenet5_mnist/` and `data/`)
from its own folder upwards and reads MNIST from `data/MNIST`, downloading it there if it is missing. It needs torch,
torchvision, numpy, pandas and openpyxl.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

## Settings

In [2]:
# ---- settings: edit here if needed ----
DATASET = "lenet5_mnist"
METHOD_KEY = "scff"                    # the weights folder is weights_<METHOD_KEY>_lenet5_mnist/
METHOD_LABEL = "SCFF"                  # the method's row label in the workbook

DEVICE = "cpu"                         # "cuda" works too, on a GPU this torch build supports
BATCH_SIZE = 1024                      # evaluation batch; lower it if memory is short
SEED = 42
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]   # input-noise variances
TEST_NOISE_SEED = SEED                 # the same noise draw at every level
NUM_CLASSES = 10


def find_project_root(start=Path.cwd()):
    """The BOLSTER folder: the first folder, from the notebook's folder upwards, that holds lenet5_mnist/ and data/."""
    for c in (start, *start.parents):
        if (c / DATASET).is_dir() and (c / "data").is_dir():
            return c
    raise FileNotFoundError(f"no folder with {DATASET}/ and data/ above {start}")


PROJECT = find_project_root()
DATA_ROOT = PROJECT / "data"
BASELINE_WEIGHTS = PROJECT / DATASET / "weights_baselines"
METHOD_WEIGHTS = PROJECT / DATASET / f"weights_{METHOD_KEY}_{DATASET}"
RESULTS_DIR = PROJECT / DATASET / f"results_{METHOD_KEY}_{DATASET}"
# no YYYYmmdd_HHMMSS stamp in the name: the plotting notebooks keep reading the reported workbook
XLSX_PATH = RESULTS_DIR / f"results_{METHOD_KEY}_{DATASET}_reproduced.xlsx"

print(f"project: {PROJECT}\ndevice: {DEVICE}")

project: /home/rahman2/GIT/BOLSTER
device: cpu


## The networks
LeNet-5 scores the Init row; the SCFF network (the same conv shapes plus SCFF's own readout) scores the SCFF rows.

In [3]:
class LeNet5(nn.Module):
    """LeNet-5 for 28x28x1 MNIST with ReLU and max-pool: conv1(6) - pool - conv2(16) - pool - conv3/C5(120) - f6(84) -
    fc(10). The layer names match the stored state dicts."""
    def __init__(self, num_classes=NUM_CLASSES, in_ch=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 6, 5, padding=2)        # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(6, 16, 5)                      # 14x14 -> 10x10
        self.conv3 = nn.Conv2d(16, 120, 5)                    # 5x5   -> 1x1  (C5)
        self.f6    = nn.Linear(120, 84)
        self.fc    = nn.Linear(84, num_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)            # -> 14x14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)            # -> 5x5
        x = F.relu(self.conv3(x))                             # -> 120x1x1
        x = x.flatten(1)                                      # -> 120
        x = F.relu(self.f6(x))
        return self.fc(x)

In [4]:
SCFF_ACTS   = ("triangle", "relu", "triangle")   # per-layer activation that carries information forward
LAYER_OUT   = (1, 2)                             # conv layers (0-indexed) whose features feed the readout
EXTRA_POOL  = 2                                  # extra avg-pool kernel/stride applied to the readout features
OUT_DROPOUT = 0.2                                # the readout's dropout: off in eval mode, kept for the key classifier.1


def stdnorm(x, dims=(1, 2, 3)):
    """Per-sample standardization: zero mean, unit std over `dims`."""
    x = x - x.mean(dim=dims, keepdim=True)
    return x / (1e-10 + x.std(dim=dims, keepdim=True))


def _triangle(x):
    """Triangle activation: subtract the per-position channel mean, then ReLU."""
    return F.relu(x - x.mean(dim=1, keepdim=True))


def _act(idx, x):
    return _triangle(x) if SCFF_ACTS[idx] == "triangle" else F.relu(x)


def _convsum(conv, x_cat):
    """Tied-weight W1=W2 form: standardize the concatenated pair, split it into the two channel halves, apply the
    same conv to each and sum."""
    x = stdnorm(x_cat, dims=(1, 2, 3))
    h = x.size(1) // 2
    return conv(x[:, :h]) + conv(x[:, h:])


class SCFFNet(nn.Module):
    """The LeNet-5 conv body (6 -> 16 -> 120) trained with SCFF, plus a linear readout over multi-layer features.
    The readout is sized from a dummy forward so that the stored state dicts load. forward(x) returns class logits."""
    def __init__(self, num_classes=NUM_CLASSES, in_ch=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 6, 5, padding=2)        # 28 -> 28 (pool -> 14)
        self.conv2 = nn.Conv2d(6, 16, 5)                      # 14 -> 10 (pool -> 5)
        self.conv3 = nn.Conv2d(16, 120, 5)                    # 5  -> 1  (pool -> 1)
        with torch.no_grad():
            feat_dim = self._extract(torch.zeros(1, in_ch, 28, 28)).shape[1]
        self.classifier = nn.Sequential(nn.Dropout(OUT_DROPOUT), nn.Linear(feat_dim, num_classes))

    @property
    def convs(self):
        return [self.conv1, self.conv2, self.conv3]

    def _extract(self, x):
        """At each layer: present x as a positive pair, conv-sum, activate, pool. The features of the layers in
        LAYER_OUT are standardized, extra-pooled, flattened and concatenated; the next layer sees the pooled stream."""
        inp, feats = x, []
        for i, conv in enumerate(self.convs):
            xin = stdnorm(inp, dims=(1, 2, 3))
            out = _convsum(conv, torch.cat((xin, xin), dim=1))
            h = F.max_pool2d(_act(i, out), 2, 2, ceil_mode=True)
            if i in LAYER_OUT:
                ep = F.avg_pool2d(h, EXTRA_POOL, EXTRA_POOL, ceil_mode=True)
                feats.append(stdnorm(ep, dims=(1, 2, 3)).flatten(1))
            inp = h
        return torch.cat(feats, dim=1)

    def forward(self, x):
        return self.classifier(self._extract(x))

## Metrics

In [5]:
def macro_prf(y_true, y_pred, num_classes=NUM_CLASSES):
    """Macro precision, recall and F1 over the classes; a class with an empty denominator contributes 0."""
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    den  = prec + rec
    f1   = np.divide(2 * prec * rec, den, out=np.zeros_like(tp), where=den > 0)
    return float(prec.mean()), float(rec.mean()), float(f1.mean())


def average_precision(is_pos, scores):
    """One-vs-rest average precision of one class (the scikit-learn definition: the sum over the distinct score
    thresholds of (R_n - R_{n-1}) P_n)."""
    n_pos = int(is_pos.sum())
    if n_pos == 0:
        return 0.0
    order = np.argsort(-scores, kind="mergesort")
    y, s = is_pos[order], scores[order]
    last = np.r_[np.where(np.diff(s))[0], y.size - 1]        # the last index of each distinct score
    tps = np.cumsum(y)[last]
    fps = (1 + last) - tps
    precision = np.r_[1.0, tps / (tps + fps)]
    recall    = np.r_[0.0, tps / n_pos]
    return float(np.sum(np.diff(recall) * precision[1:]))


@torch.no_grad()
def eval_metrics(state_dict, x, y, model_cls=LeNet5):
    """accuracy, macro precision / recall / F1 and mAP of the model in state_dict on (x, y). The argmax gives the
    predictions, the softmax the scores for mAP."""
    model = model_cls().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    probs, preds = [], []
    for i in range(0, y.numel(), BATCH_SIZE):
        logits = model(x[i:i + BATCH_SIZE].to(DEVICE))
        probs.append(F.softmax(logits, dim=1).cpu())
        preds.append(logits.argmax(1).cpu())
    probs, y_pred, y_true = torch.cat(probs).numpy(), torch.cat(preds).numpy(), y.numpy()
    precision, recall, f1 = macro_prf(y_true, y_pred)
    mAP = float(np.mean([average_precision((y_true == c).astype(np.int64), probs[:, c]) for c in range(NUM_CLASSES)]))
    return {"accuracy": float((y_pred == y_true).mean()), "precision": precision, "recall": recall, "f1": f1,
            "map": mAP}

## The test set

In [6]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


def stack(ds):
    xs, ys = zip(*((x, y) for x, y in ds))
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)


def add_noise(x, var, seed):
    """x plus Gaussian noise of variance var, drawn on the CPU from a generator seeded with seed."""
    if var <= 0:
        return x
    g = torch.Generator().manual_seed(seed)
    return x + torch.randn(x.shape, generator=g) * float(np.sqrt(var))


x_test, y_test = stack(datasets.MNIST(str(DATA_ROOT), train=False, download=True, transform=tfm))
print(f"test set: the official MNIST test set, {y_test.numel():,} images")

test set: the official MNIST test set, 10,000 images


## Scores and workbook
Which weight file feeds which row; every row is scored on that level's noisy test set, then the workbook is written.

In [7]:
ROWS = [                               # (row label, weight file for noise variance v, network), in the workbook's order
    ("Init (no retrain)", lambda v: BASELINE_WEIGHTS / "full_retrain_var_0.0.pth", LeNet5),
    (METHOD_LABEL, lambda v: METHOD_WEIGHTS / f"{METHOD_KEY}_var_{v}.pth", SCFFNet),
]
METRIC_COLS = ["accuracy", "precision", "recall", "f1", "map"]


def load_state(path):
    return torch.load(path, map_location="cpu", weights_only=True)


records = []
for v in NOISE_LEVELS:
    level = f"{v:.0%}"
    x_noisy = add_noise(x_test, v, seed=TEST_NOISE_SEED)
    for label, path_of, model_cls in ROWS:
        path = path_of(v)
        r = eval_metrics(load_state(path), x_noisy, y_test, model_cls)
        records.append({"noise_level": level, "method": label, **{m: round(r[m], 6) for m in METRIC_COLS}})
        print(f"{level:>5}  {label:<20} acc {r['accuracy']:.6f}  f1 {r['f1']:.6f}  map {r['map']:.6f}   <- {path.name}")

results = pd.DataFrame(records, columns=["noise_level", "method", *METRIC_COLS])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_excel(XLSX_PATH, sheet_name="long", index=False)
print(f"\nwrote {XLSX_PATH.relative_to(PROJECT)}")

  10%  Init (no retrain)    acc 0.976400  f1 0.976239  map 0.996373   <- full_retrain_var_0.0.pth


  10%  SCFF                 acc 0.947100  f1 0.946537  map 0.985741   <- scff_var_0.1.pth


  20%  Init (no retrain)    acc 0.972900  f1 0.972769  map 0.995223   <- full_retrain_var_0.0.pth


  20%  SCFF                 acc 0.923600  f1 0.922760  map 0.975834   <- scff_var_0.2.pth


  30%  Init (no retrain)    acc 0.967100  f1 0.966848  map 0.993665   <- full_retrain_var_0.0.pth


  30%  SCFF                 acc 0.884500  f1 0.882909  map 0.947615   <- scff_var_0.3.pth


... (10 outputs left out)

  90%  Init (no retrain)    acc 0.898300  f1 0.899550  map 0.972855   <- full_retrain_var_0.0.pth


  90%  SCFF                 acc 0.699400  f1 0.694679  map 0.772331   <- scff_var_0.9.pth


 100%  Init (no retrain)    acc 0.879400  f1 0.881177  map 0.967120   <- full_retrain_var_0.0.pth


 100%  SCFF                 acc 0.657200  f1 0.651484  map 0.723003   <- scff_var_1.0.pth

wrote lenet5_mnist/results_scff_lenet5_mnist/results_scff_lenet5_mnist_reproduced.xlsx
